# 04 · Clima

**Preguntas:**
- ¿Cómo llueve en la zona: cuánto, en qué meses y a qué hora?
- ¿Cuánto sube la turbidez con la lluvia, y con cuánto retraso llega a cada sensor?
- ¿Qué ventana de lluvia acumulada explica mejor la turbidez?
- ¿Qué pasa con el cloro, el pH y la temperatura cuando llueve?
- ¿Cuánto le serviría al pronóstico conocer la lluvia que **todavía no ha caído**?

La lluvia (`lluvia_mm_h`) es la medida hora por hora por el Instituto Meteorológico Nacional (IMN)
en la estación 69-701 ITCR, San Carlos (10°21' N, 84°31' O, 187 msnm). Cada lectura de 15 min trae
la lluvia de la última hora completa: la de las 14:15 trae la que cayó de 13:00 a 14:00. Así, en
cada hora solo se usa lluvia que ya cayó.

Como en el notebook 03, se trabaja con **promedios horarios** de las lecturas válidas y, para medir
la respuesta a la lluvia, solo con la **operación normal** (sin evento visible en el sensor): la
turbidez de un evento en la fuente no se debe confundir con la de un aguacero.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))          # por si calidad_agua no está instalado (pip install -e .)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from calidad_agua import carga, calidad, metas
from calidad_agua.graficos import estilo, guardar, COLORES, GRIS

estilo()
pd.set_option("display.precision", 3)
pd.set_option("display.width", 200)

t15 = calidad.lecturas_validas(carga.cargar_todo())
S = carga.sensores(t15)
h = t15.drop(columns="particion").resample("1h").mean()
evento = t15[[f"evento_{n}" for n in S]].resample("1h").max()
particion = t15["particion"].resample("1h").first()
lluvia = t15["lluvia_mm_h"].resample("1h").mean()          # mm de la última hora completa

def normal(p, n):
    # serie horaria de un sensor, solo en operación normal
    return h[f"{p}_{n}"].where(evento[f"evento_{n}"] == 0)

REPRES = {"J280": "salida de la fuente", "J224": "intermedio", "J144": "extremo de la red"}
ETIQ = {"CL2": "Cloro libre (mg/L)", "TEMP": "Temperatura (°C)", "TURB": "Turbidez (UNT)", "PH": "pH"}
print(f"{len(h):,} horas · {lluvia.index[0].date()} a {lluvia.index[-1].date()}")

17,520 horas · 2024-09-01 a 2026-08-31


## 1. Cómo llueve en la zona

In [2]:
dia = lluvia.resample("1D").sum()
anios = (lluvia.index[-1] - lluvia.index[0]).days / 365.25
resumen = pd.Series({
    "lluvia por año (mm)": lluvia.sum() / anios,
    "días con al menos 1 mm (%)": 100 * (dia >= 1).mean(),
    "horas con lluvia (%)": 100 * (lluvia > 0).mean(),
    "día más lluvioso (mm)": dia.max(),
    "hora más lluviosa (mm)": lluvia.max(),
}).round(1)
print(f"Día más lluvioso: {dia.idxmax().date()} · hora más lluviosa: {lluvia.idxmax()}")
resumen.to_frame("valor")

Día más lluvioso: 2025-06-24 · hora más lluviosa: 2024-10-17 19:00:00


,valor
lluvia por año (mm),4597.3
días con al menos 1 mm (%),64.7
horas con lluvia (%),22.0
día más lluvioso (mm),202.0
hora más lluviosa (mm),88.2


In [3]:
mes = lluvia.resample("MS").sum()
# la lluvia de la lectura de las h:00 es la de (h-1):00 a h:00: se asigna a la hora en que empezó
por_hora = lluvia.groupby((lluvia.index - pd.Timedelta(hours=1)).hour).sum() / anios

fig, axes = plt.subplots(1, 2, figsize=(11, 3.2), gridspec_kw={"width_ratios": [2.2, 1]})
axes[0].bar(mes.index, mes.values, width=25, color=COLORES[0])
axes[0].xaxis.set_major_formatter(mdates.DateFormatter("%b\n%Y"))
axes[0].set_ylabel("mm en el mes")
axes[0].set_title("Lluvia de cada mes")
axes[1].bar(por_hora.index, por_hora.values, width=0.8, color=COLORES[0])
axes[1].set_xticks([0, 6, 12, 18, 23]); axes[1].set_xlabel("hora del día")
axes[1].set_ylabel("mm por año")
axes[1].set_title("¿A qué hora llueve?")
fig.tight_layout()
guardar(fig, "04_lluvia"); plt.show()

## 2. La turbidez y la lluvia: retraso y ventana

**Retraso.** Correlación (Spearman) entre la lluvia de cada hora y la turbidez *k* horas después,
en la salida de la fuente, un sensor intermedio y el extremo de la red. El retraso con la
correlación más alta dice cuánto tarda el agua turbia en llegar.

**Ventana.** Correlación entre la turbidez de la fuente y la lluvia acumulada en las últimas *k*
horas: dice qué tanta "memoria" de lluvia hay que darle al modelo.

In [4]:
retrasos = np.arange(0, 73)
cc = pd.DataFrame({n: [normal("TURB", n).corr(lluvia.shift(k), method="spearman") for k in retrasos]
                   for n in REPRES}, index=retrasos)
ventanas = [1, 3, 6, 12, 24, 48, 72, 168]
cv = pd.Series({k: normal("TURB", "J280").corr(lluvia.rolling(k, min_periods=1).sum(), method="spearman")
                for k in ventanas}, name="correlación con la turbidez de la fuente")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
for k, (n, nombre) in enumerate(REPRES.items()):
    axes[0].plot(cc.index, cc[n], color=COLORES[k], label=f"{n} ({nombre})")
    axes[0].plot(cc[n].idxmax(), cc[n].max(), "o", ms=5, color=COLORES[k])
axes[0].set_xlabel("horas después de la lluvia"); axes[0].set_ylabel("correlación")
axes[0].set_xticks([0, 6, 12, 24, 36, 48, 72]); axes[0].legend()
axes[0].set_title("La turbidez llega con retraso")
axes[1].plot(range(len(cv)), cv.values, "o-", color=COLORES[0])
axes[1].set_xticks(range(len(cv))); axes[1].set_xticklabels([f"{k} h" for k in cv.index])
axes[1].set_xlabel("lluvia acumulada en las últimas…"); axes[1].set_ylabel("correlación")
axes[1].set_title("Turbidez de la fuente y lluvia acumulada")
fig.tight_layout()
guardar(fig, "04_retraso_ventana"); plt.show()

pd.DataFrame({"retraso con más correlación (h)": cc.idxmax(), "correlación": cc.max()}).T.round(3)

,J280,J224,J144
retraso con más correlación (h),3.000,8.000,20.000
correlación,0.504,0.486,0.508


In [5]:
cv.round(3).to_frame().T

,1,3,6,12,24,48,72,168
correlación con la turbidez de la fuente,0.38,0.532,0.664,0.815,0.905,0.836,0.73,0.492


## 3. Qué pasa después de un aguacero

Se toman los **aguaceros** que llegan después de al menos 24 h casi secas (menos de 2 mm) y dejan
15 mm o más en 6 horas, y se mira cómo cambia cada parámetro respecto a su valor de las 3 horas
previas, desde 6 h antes hasta 48 h después. Se muestra la mediana de todos los aguaceros. Así se
ve la respuesta típica sin que la tapen otras lluvias.

In [6]:
en6h = lluvia.rolling(6).sum().shift(-5)                     # lluvia de esa hora y las 5 siguientes
candidatos = lluvia.index[(lluvia >= 1) & (lluvia.rolling(24).sum().shift(1) < 2) & (en6h >= 15)]
aguaceros = []
for t in candidatos:                                          # uno por tormenta
    if not aguaceros or t - aguaceros[-1] > pd.Timedelta(hours=48):
        aguaceros.append(t)
rel = np.arange(-6, 49)

def respuesta(p, n):
    s = normal(p, n)
    filas = []
    for t in aguaceros:
        antes = s.reindex(t + pd.to_timedelta(np.arange(-3, 0), unit="h")).mean()
        filas.append(s.reindex(t + pd.to_timedelta(rel, unit="h")).values - antes)
    return np.nanmedian(np.array(filas), axis=0)

print(f"{len(aguaceros)} aguaceros · lluvia mediana en 6 h: {np.median([en6h[t] for t in aguaceros]):.0f} mm")
resp = {(p, n): respuesta(p, n) for p in ["TURB", "CL2", "PH", "TEMP"] for n in REPRES}

fig, axes = plt.subplots(1, 4, figsize=(12, 3.2), sharex=True)
for ax, p in zip(axes, ["TURB", "CL2", "PH", "TEMP"]):
    for k, n in enumerate(REPRES):
        ax.plot(rel, resp[p, n], color=COLORES[k], label=n)
    ax.axvline(0, color=GRIS, lw=1, ls=":"); ax.axhline(0, color=GRIS, lw=0.8)
    ax.set_title(f"Δ {ETIQ[p]}", fontsize=10); ax.set_xticks([0, 12, 24, 36, 48])
    ax.set_xlabel("horas desde el inicio del aguacero")
axes[0].legend()
fig.tight_layout()
guardar(fig, "04_respuesta_aguacero"); plt.show()

tabla_resp = pd.DataFrame({
    (p, n): {"cambio máximo": resp[p, n][np.nanargmax(np.abs(resp[p, n]))],
             "horas hasta el máximo": rel[np.nanargmax(np.abs(resp[p, n]))],
             "cambio a las 24 h": resp[p, n][rel == 24][0]}
    for p in ["TURB", "CL2", "PH", "TEMP"] for n in REPRES}).T
tabla_resp.round(3)

45 aguaceros · lluvia mediana en 6 h: 30 mm


cambio máximo  horas hasta el máximo  cambio a las 24 h
TURB J280          1.655                    6.0              0.647
     J224          1.533                   10.0              0.812
     J144          1.556                   25.0              1.442
CL2  J280          0.083                    8.0              0.052
     J224          0.096                   10.0              0.059
     J144          0.081                   32.0              0.050
PH   J280         -0.175                    4.0             -0.127
     J224         -0.177                    8.0             -0.127
     J144         -0.185                   23.0             -0.162
TEMP J280         -1.117                   12.0             -0.587
     J224         -0.975                   12.0             -0.558
     J144         -1.092                   24.0             -1.092

El cambio de temperatura incluye el ciclo diario: casi todos los aguaceros son de tarde, y después
viene la noche. Aun así, la caída es más del doble de la oscilación diaria normal (unos 0,4 °C).

### Las primeras lluvias arrastran más

Después de un periodo seco, la primera lluvia lava más sedimento hacia la fuente. Se compara cuánto
sube la turbidez de la fuente por cada mm, en días con 10 mm o más, según cuánto llovió en los 30
días anteriores.

In [7]:
dia_tb = normal("TURB", "J280")
lav = pd.DataFrame({"lluvia del día (mm)": dia,
                    "subida (UNT)": dia_tb.resample("1D").max() - dia_tb.resample("1D").min().shift(1),
                    "lluvia de los 30 días previos (mm)": dia.rolling(30).sum().shift(1)}).dropna()
lav = lav[lav["lluvia del día (mm)"] >= 10]
lav["UNT por mm"] = lav["subida (UNT)"] / lav["lluvia del día (mm)"]
lav["30 días previos"] = pd.cut(lav["lluvia de los 30 días previos (mm)"], [0, 100, 200, 400, 5000],
                                labels=["menos de 100 mm", "100–200 mm", "200–400 mm", "más de 400 mm"])
lavado = lav.groupby("30 días previos", observed=True)["UNT por mm"].agg(["median", "count"])
lavado.columns = ["UNT por mm (mediana)", "días"]

fig, ax = plt.subplots(figsize=(6.5, 3))
barras = ax.bar(range(len(lavado)), lavado.iloc[:, 0], color=COLORES[0], width=0.6)
for b_, v_, n_ in zip(barras, lavado.iloc[:, 0], lavado.iloc[:, 1]):
    ax.text(b_.get_x() + b_.get_width() / 2, v_, f"{v_:.3f}\n({n_} días)", ha="center", va="bottom", fontsize=7)
ax.set_xticks(range(len(lavado))); ax.set_xticklabels(lavado.index)
ax.set_xlabel("lluvia de los 30 días anteriores"); ax.set_ylabel("UNT por mm")
ax.set_ylim(0, lavado.iloc[:, 0].max() * 1.3)
ax.set_title("Cuanto más seco el mes anterior, más turbidez por mm")
guardar(fig, "04_lavado"); plt.show()
lavado.round(4)

,UNT por mm (mediana),días
30 días previos,,
menos de 100 mm,0.075,9
100–200 mm,0.068,32
200–400 mm,0.052,59
más de 400 mm,0.046,130


## 4. ¿Cuánto serviría conocer la lluvia que viene?

El pronóstico a 24 h se hace en la hora *t* para la hora *t* + 24. La lluvia que cae **entre esas dos
horas** todavía no está en los datos. ¿Cuánto falla la línea base ("igual que ayer") según cuánto
llueve en ese intervalo? Se usa la operación normal de validación y prueba, en los 21 sensores.

In [8]:
llueve_24h = lluvia.rolling(24).sum()          # lluvia de las 24 h que terminan en la hora pronosticada
filas = []
for n in S:
    y = h[f"TURB_{n}"]
    m = (evento[f"evento_{n}"] == 0) & particion.isin(["validacion", "prueba"])
    d = pd.DataFrame({"obs": y, "base": y.shift(24), "lluvia": llueve_24h})[m].dropna()
    d["acierto"] = metas.acierto("TURB", d["obs"].values, d["base"].values)
    filas.append(d)
d = pd.concat(filas)
d["lluvia entre el pronóstico y la hora pronosticada"] = pd.cut(
    d["lluvia"], [-0.1, 0.5, 10, 30, 10_000], labels=["sin lluvia", "0,5–10 mm", "10–30 mm", "más de 30 mm"])
acierto_lluvia = d.groupby("lluvia entre el pronóstico y la hora pronosticada", observed=True)["acierto"] \
                  .agg(["mean", "size"]).rename(columns={"mean": "% acierto línea base", "size": "pronósticos"})
acierto_lluvia["% acierto línea base"] *= 100
acierto_lluvia["% de los pronósticos"] = 100 * acierto_lluvia["pronósticos"] / acierto_lluvia["pronósticos"].sum()

fig, ax = plt.subplots(figsize=(6.5, 3))
barras = ax.bar(range(len(acierto_lluvia)), acierto_lluvia["% acierto línea base"], color=COLORES[0], width=0.6)
for b_, v_ in zip(barras, acierto_lluvia["% acierto línea base"]):
    ax.text(b_.get_x() + b_.get_width() / 2, v_ + 1, f"{v_:.0f} %", ha="center", fontsize=8)
ax.axhline(metas.META_PCT, color=COLORES[7], ls="--", lw=1)
ax.set_xticks(range(len(acierto_lluvia))); ax.set_xticklabels(acierto_lluvia.index)
ax.set_ylim(0, 100); ax.set_ylabel("% dentro de la tolerancia")
ax.set_title("Turbidez: la línea base falla cuando llueve")
guardar(fig, "04_acierto_segun_lluvia"); plt.show()
acierto_lluvia.round(1)

,% acierto línea base,pronósticos,% de los pronósticos
lluvia entre el pronóstico y la hora pronosticada,,,
sin lluvia,73.7,42002,36.8
"0,5–10 mm",50.9,39503,34.6
10–30 mm,28.9,22383,19.6
más de 30 mm,20.1,10219,9.0


In [9]:
# ¿El cambio de turbidez en las próximas 24 h se relaciona más con la lluvia que ya cayó o con la que viene?
tb = normal("TURB", "J280")
cambio = tb.shift(-24) - tb
pd.Series({"lluvia de las últimas 24 h": cambio.corr(lluvia.rolling(24).sum(), method="spearman"),
           "lluvia de las próximas 24 h": cambio.corr(lluvia.rolling(24).sum().shift(-24), method="spearman")},
          name="correlación con el cambio de turbidez a 24 h (fuente)").round(3).to_frame()

,correlación con el cambio de turbidez a 24 h (fuente)
lluvia de las últimas 24 h,-0.450
lluvia de las próximas 24 h,0.399


## 5. ¿Qué tan bueno es el pronóstico de lluvia?

Para usar la lluvia que viene hace falta un pronóstico, y para entrenar, saber qué decía ese
pronóstico **en su momento**. La Previous Runs API de Open-Meteo guarda, para cada hora, la lluvia
que pronosticaba cada modelo meteorológico 24 h antes (`clima/pronostico_lluvia_modelos.csv`). Se
comparan cuatro modelos globales y su promedio contra la lluvia medida por el IMN:

- **correlación** (Spearman) entre la lluvia pronosticada y la real de las 24 h siguientes a cada hora;
- **sesgo**: cuántas veces la lluvia real es la pronosticada, por trimestre. Si cambia mucho, un
  modelo entrenado con un año no sirve para el siguiente;
- **días con 10 mm o más**: cuántos avisó el pronóstico (5 mm o más en el día) y cuántos avisos fueron falsos.

In [10]:
from calidad_agua import clima
NOMBRES = {"ecmwf_ifs025": "ECMWF IFS", "gfs_seamless": "GFS", "icon_seamless": "ICON", "gem_seamless": "GEM"}
modelos_pron = clima.leer().reindex(lluvia.index)
modelos_pron["promedio de los 4"] = modelos_pron[clima.MODELOS].mean(axis=1)
real_24h = lluvia[::-1].rolling(24, min_periods=1).sum()[::-1].shift(-1)      # lluvia de t+1 … t+24
llueve_dia = dia >= 10
filas, sesgo = {}, {}
for col in modelos_pron.columns:
    pr_24h = clima.ventanas(modelos_pron[col], lluvia.index).sum(axis=1)
    x = pd.DataFrame({"real": real_24h, "pron": pr_24h}).dropna()
    trimestre = x.groupby(pd.Grouper(freq="QS"))
    sesgo[NOMBRES.get(col, col)] = trimestre["real"].mean() / trimestre["pron"].mean()
    aviso = modelos_pron[col].resample("1D").sum() >= 5
    filas[NOMBRES.get(col, col)] = {
        "correlación con la lluvia real (24 h)": x["real"].corr(x["pron"], method="spearman"),
        "lluvia real / pronosticada (total)": x["real"].sum() / x["pron"].sum(),
        "sesgo mínimo por trimestre": sesgo[NOMBRES.get(col, col)].min(),
        "sesgo máximo por trimestre": sesgo[NOMBRES.get(col, col)].max(),
        "días ≥ 10 mm avisados (%)": 100 * (aviso & llueve_dia).sum() / llueve_dia.sum(),
        "avisos falsos (días)": int((aviso & ~llueve_dia).sum()),
    }
calidad_pron = pd.DataFrame(filas).T
fig, axes = plt.subplots(1, 2, figsize=(11, 3.2), gridspec_kw={"width_ratios": [1, 1.6]})
c = calidad_pron["correlación con la lluvia real (24 h)"]
axes[0].barh(range(len(c)), c.values, color=[COLORES[0]] * (len(c) - 1) + [COLORES[1]], height=0.6)
axes[0].set_yticks(range(len(c))); axes[0].set_yticklabels(c.index); axes[0].invert_yaxis()
for k, v in enumerate(c.values):
    axes[0].text(v + 0.01, k, f"{v:.2f}", va="center", fontsize=8)
axes[0].set_xlim(0, 0.8); axes[0].set_xlabel("correlación con la lluvia real de las 24 h siguientes")
axes[0].set_title("¿Qué pronóstico acierta más?")
for k, (nombre, s) in enumerate(sesgo.items()):
    axes[1].plot(s.index, s.values, "o-", ms=3, color=COLORES[k], label=nombre)
axes[1].axhline(1, color=GRIS, lw=0.8)
axes[1].set_ylabel("lluvia real / pronosticada"); axes[1].legend(ncol=3)
axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%b\n%Y"))
axes[1].set_title("Sesgo de cada pronóstico por trimestre")
fig.tight_layout()
guardar(fig, "04_pronosticos_lluvia"); plt.show()
calidad_pron.round(2)

,correlación con la lluvia real (24 h),lluvia real / pronosticada (total),sesgo mínimo por trimestre,sesgo máximo por trimestre,días ≥ 10 mm avisados (%),avisos falsos (días)
ECMWF IFS,0.59,1.55,1.08,2.13,83.47,208.0
GFS,0.57,1.88,1.24,3.83,67.77,147.0
ICON,0.50,3.58,2.10,7.83,41.74,80.0
GEM,0.54,0.76,0.52,1.09,91.32,298.0
promedio de los 4,0.65,1.44,0.97,1.91,92.56,231.0


Una primera prueba con la opción por defecto de Open-Meteo ("best_match", que elige el modelo
automáticamente) dio una correlación de 0,49 y un sesgo que iba de 0,8 a 7,8 según el trimestre:
cambia de modelo con el tiempo y su sesgo cambia con él. Por eso se usan modelos fijos.

## 6. Conclusiones e implicaciones para el modelo

**Lo que dicen los datos**

1. **La zona es muy lluviosa, y llueve sobre todo de tarde.** Unos 4 600 mm al año; llueve en el
   65 % de los días y en el 22 % de las horas. Los meses más secos son marzo y abril (de 44 a
   120 mm); el resto del año la lluvia cambia mucho de un año a otro, y en la mitad de los meses
   pasa de 400 mm (hasta 762 mm en noviembre de 2024). Los aguaceros fuertes caen entre las 15:00
   y las 19:00: en esas horas cae unas cuatro veces más lluvia que a media mañana. El día
   más lluvioso dejó 202 mm (24 de junio de 2025) y la hora más lluviosa, 88 mm.
2. **La turbidez responde a la lluvia con retraso, y el retraso crece hacia el extremo de la red.**
   La correlación más alta entre la lluvia de una hora y la turbidez está 3 h después en la
   fuente, 8 h después en J224 y 20 h después en J144. Es el tiempo que tarda el agua turbia en
   llegar a la fuente más el tiempo de viaje por la red (3 y 15 h, notebook 03).
3. **La lluvia de las últimas 24 h es la que mejor explica la turbidez de la fuente** (correlación
   0,91). Ventanas más cortas se quedan sin la lluvia que todavía está llegando (0,66 con 6 h) y
   más largas mezclan lluvias que ya pasaron (0,73 con 72 h).
4. **Un aguacero típico (30 mm en 6 h, tras un día seco) mueve todos los parámetros:**
   - la turbidez sube unos 1,6 UNT: en la fuente llega al máximo a las 6 h, en J224 a las 10 h y
     en J144 a las 25 h, y un día después sigue 0,6 UNT por encima en la fuente;
   - el cloro **sube** unos 0,08 mg/L, porque la planta aumenta la dosis cuando el agua llega más
     turbia;
   - el pH baja unos 0,18;
   - la temperatura baja cerca de 1 °C (incluye el enfriamiento de la noche, pero es más del doble
     de la oscilación diaria normal).
5. **Las primeras lluvias arrastran más.** Por cada mm, la turbidez de la fuente sube 0,075 UNT si
   el mes anterior fue seco (menos de 100 mm), y 0,046 UNT si fue muy lluvioso (más de 400 mm): un
   60 % más después de la sequía.
6. **Lo que más falta es la lluvia que todavía no ha caído.** En validación y prueba, la línea base
   de turbidez acierta el 74 % cuando no llueve entre el pronóstico y la hora pronosticada, el
   51 % con 0,5 a 10 mm, el 29 % con 10 a 30 mm y el 20 % con más de 30 mm. Y en el 63 % de los
   pronósticos llueve algo en ese intervalo. El cambio de turbidez a 24 h se relaciona tanto con
   la lluvia que ya cayó (−0,45: la turbidez baja cuando escampa) como con la que viene (+0,40:
   sube si va a llover). El modelo ya ve la primera, pero no la segunda.
7. **El mejor pronóstico de lluvia es el promedio de cuatro modelos, y aun así acierta a medias.**
   Su correlación con la lluvia real de las 24 h siguientes es 0,65, contra 0,50–0,59 de cada modelo
   por separado, y su sesgo es estable (la lluvia real es entre 1,0 y 1,9 veces la pronosticada,
   según el trimestre). Avisa el 93 % de los días con 10 mm o más, pero con muchos avisos falsos
   (231 días). La opción por defecto de Open-Meteo ("best_match") es peor (0,49) y su sesgo cambia
   de 0,8 a 7,8 veces con el tiempo.

**Qué implica para el modelo**

- **Variables de lluvia que ya cayó:** mantener la lluvia acumulada de 24 h, que es la que mejor
  explica la turbidez. También se probaron 12 h, 48 h y 30 días (para el lavado), pero no mejoraron
  el pronóstico (notebook 06).
- **Pronóstico de lluvia para las próximas 24 h:** es lo que más le puede sumar a la turbidez, y
  también al pH, al cloro y a la temperatura, que se mueven con cada aguacero. Conviene darlo por
  tramos (próximas 0–6, 6–12 y 12–24 h), porque una lluvia a las 2 h y otra a las 20 h afectan
  distinto a la hora pronosticada, y más aún según el tiempo de viaje de cada sensor.
- **Entrenar con pronósticos, no con la lluvia que de verdad cayó.** Si el modelo aprende con la
  lluvia observada, aprende a confiar en un pronóstico perfecto que en operación no existe. Hay
  que entrenarlo con lo que un pronóstico habría dicho en cada momento: los pronósticos pasados
  que guarda Open-Meteo (Previous Runs API) para las coordenadas de la estación, con el promedio de
  los cuatro modelos y no con "best_match". La lluvia observada sirve como techo: dice cuánto se
  ganaría con un pronóstico perfecto.
- **Los sensores lejanos ya tienen parte de la respuesta en los datos.** En J144 la turbidez
  llega unas 20 h después de la lluvia, así que buena parte de lo que pasará en las próximas 24 h
  ya cayó. Ahí el pronóstico de lluvia aporta menos que en la fuente.
